# Smart MCQ Solver - DL & GenAI Project

# 1. Environment Setup

Import all the required libraries that will be used throughout this project.

In [1]:
import os
import re
import json
import warnings

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity

print("Environment Ready ✅")

Environment Ready ✅


# 2. Configuration

Define project constants and file paths used throughout the notebook.

In [2]:
SEED = 42
VAL_SIZE = 0.2

TRAIN_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
TEST_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"
SAMPLE_SUBMISSION_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv"

OPTIONS = ["A", "B", "C", "D", "E"]

# 3. Load the Dataset

Load the training and test datasets and inspect their basic structure before performing any preprocessing or modeling.

In [3]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)

print(f"Train Shape: {train.shape}")
print(f"Test Shape: {test.shape}")

train.head()

Train Shape: (2000, 8)
Test Shape: (500, 7)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


# 4. Dataset Overview

Understand the structure, data types, and completeness of the dataset.

In [4]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      2000 non-null   int64 
 1   prompt  2000 non-null   object
 2   A       2000 non-null   object
 3   B       2000 non-null   object
 4   C       2000 non-null   object
 5   D       2000 non-null   object
 6   E       2000 non-null   object
 7   answer  2000 non-null   object
dtypes: int64(1), object(7)
memory usage: 125.1+ KB


# 5. NLP 

Machine learning models cannot understand raw text. Before applying any learning algorithm, textual data must be converted into numerical representations. In this milestone, we build progressively better text representations, starting from TF-IDF and Word2Vec.

In [5]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()

text_columns = ["prompt", "A", "B", "C", "D", "E"]

for col in text_columns:
    train[col] = train[col].apply(clean_text)
    test[col] = test[col].apply(clean_text)

## Tokenization

Tokenization is the process of breaking raw text into smaller units called tokens. These tokens form the basic input for almost every NLP model, from TF-IDF to modern Transformer architectures.

In [6]:
sample_text = train.loc[0, "prompt"]

tokens = sample_text.split()

print("Original Text:\n")
print(sample_text)

print("\nTokens:\n")
print(tokens)

Original Text:

pick the best possible answer: what is martin heidegger's view on the relationship between time and human existence? among the listed options.

Tokens:

['pick', 'the', 'best', 'possible', 'answer:', 'what', 'is', 'martin', "heidegger's", 'view', 'on', 'the', 'relationship', 'between', 'time', 'and', 'human', 'existence?', 'among', 'the', 'listed', 'options.']


## TF-IDF (Term Frequency - Inverse Document Frequency)

Bag of Words treats every word as equally important. However, common words such as "is", "the", and "of" appear in almost every document and contribute very little information.

TF-IDF improves upon Bag of Words by assigning higher weights to important words while reducing the influence of very common words.

### TF-IDF Vectorization

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()

train_corpus = (
    train["prompt"] + " " +
    train["A"] + " " +
    train["B"] + " " +
    train["C"] + " " +
    train["D"] + " " +
    train["E"]
)

tfidf_matrix = tfidf.fit_transform(train_corpus)

print("TF-IDF Matrix Shape:", tfidf_matrix.shape)

TF-IDF Matrix Shape: (2000, 2940)


### Similarity using TF-IDF

Once every document has been converted into TF-IDF vectors, we can compare two pieces of text by measuring how similar their vectors are. Cosine Similarity is the most commonly used metric for this purpose.

In [8]:
sample_prompt = train.loc[0, "prompt"]

option_vectors = tfidf.transform(train.loc[[0], ["A", "B", "C", "D", "E"]].values.flatten())
prompt_vector = tfidf.transform([sample_prompt])

scores = cosine_similarity(prompt_vector, option_vectors).flatten()

for option, score in zip(OPTIONS, scores):
    print(f"{option}: {score:.4f}")

A: 0.2619
B: 0.2953
C: 0.5777
D: 0.5238
E: 0.2332


## Word2Vec Embeddings

Unlike TF-IDF, which represents words using frequency statistics, Word2Vec learns dense vector representations where semantically similar words are placed closer together in the embedding space.

For this project, Word2Vec serves as a conceptual improvement over TF-IDF before moving to Transformer-based embeddings.

In [9]:
from gensim.models import Word2Vec

sentences = [text.split() for text in train["prompt"]]

word2vec = Word2Vec(
    sentences,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    seed=SEED
)

print(word2vec.wv.most_similar("following", topn=5))

[('correct?', 0.9940389394760132), ('accurately', 0.9925872087478638), ('describes', 0.991331934928894), ('of', 0.9905304908752441), ('definition', 0.9869868159294128)]


## Evaluation Metric - Mean Average Precision @ 3 (MAP@3)

The competition is evaluated using MAP@3 instead of accuracy. Since each prediction consists of the top three ranked answer choices, this metric rewards models that rank the correct answer higher.

In [10]:
def average_precision_at_3(actual, predicted):
    if actual in predicted[:3]:
        return 1 / (predicted[:3].index(actual) + 1)
    return 0

def map_at_3(actuals, predictions):
    scores = [
        average_precision_at_3(a, p)
        for a, p in zip(actuals, predictions)
    ]
    return np.mean(scores)

# 6. Transformer-based NLP

Classical embedding methods such as TF-IDF and Word2Vec have significant limitations in understanding context and semantics. Transformer models overcome these limitations by learning contextual representations of language through the attention mechanism.

## Context-Aware Sentence Embeddings

Sentence Transformers convert complete sentences into dense vector representations while preserving semantic meaning. These embeddings can be compared using cosine similarity for semantic retrieval and ranking.

In [11]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

sample_sentences = train.loc[:4, "prompt"].tolist()

embeddings = embedding_model.encode(sample_sentences)

print("Embedding Shape:", embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding Shape: (5, 384)


### Semantic Similarity using Sentence Transformers

Sentence embeddings capture the semantic meaning of text. By comparing the embedding of a question with the embeddings of its answer choices, we can rank the options based on semantic similarity.

In [12]:
sample = train.iloc[0]

texts = [sample["prompt"]] + [sample[option] for option in OPTIONS]

embeddings = embedding_model.encode(texts)

prompt_embedding = embeddings[0]
option_embeddings = embeddings[1:]

scores = cosine_similarity(
    [prompt_embedding],
    option_embeddings
).flatten()

for option, score in zip(OPTIONS, scores):
    print(f"{option}: {score:.4f}")

A: 0.7308
B: 0.7658
C: 0.7940
D: 0.7701
E: 0.7323


### Zero-shot Classification

Zero-shot classification uses a pretrained Natural Language Inference (NLI) model to determine how well each answer option matches the given question, without requiring any additional training on the dataset.

Each answer choice is scored independently, and the highest scoring options are selected as the predicted answers.

In [13]:
from transformers import pipeline

zero_shot = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

sample = train.iloc[0]

result = zero_shot(
    sample["prompt"],
    candidate_labels=[sample[option] for option in OPTIONS],
    multi_label=False
)

for label, score in zip(result["labels"], result["scores"]):
    print(f"{score:.4f}  {label}")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

0.4777  martin heidegger believes that humans do not exist inside time, but that they are time. the relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.
0.2054  martin heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end. the relationship to the past involves acknowledging it as a historical era, and the relationship to the future involves creating a world that will endure beyond one's own time.
0.1274  martin heidegger believes that the relationship between time and human existence is cyclical. the past and present are interconnected and the future is predetermined. human beings do not have free will.
0.1134  martin heidegger does not believe in the existence of time or that it has any effect on human consciousness. the relationship to the past and the future is insignificant, and human existence is sole